# 05c — Unmatched eligible-pool panels

Builds the `*_unmatched_cs_ready.csv` panels: treated firms + all structurally eligible controls, **no matching**. Consumed by the 06/07 `*_unmatched.Rmd` scripts, which do doubly-robust covariate adjustment via `xformla`.

**Note on size/runtime:** the eligible pool is large (tens of thousands of controls), so the unmatched C&S run in R uses a reduced bootstrap; see the `.Rmd`.

In [ ]:
import numpy as np
import pandas as pd

from ipcei_spillovers.io import load_df
from ipcei_spillovers.config_paths import R_DATA_DIR
from ipcei_spillovers.matching import eligible_pool, entropy_balance
from ipcei_spillovers.panels import build_cs_panel

R_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Same inputs as the PSM notebook (05a)
firm_master_ipcei = load_df(
    "interim", "combined",
    "firm_master_ipcei_corrected_extended_jan26.csv", low_memory=False
)
partnership_master_ipcei = load_df(
    "interim", "combined",
    "partnership_master_ipcei_jan26.csv", low_memory=False
)
firm_month_outcomes = load_df(
    "interim", "combined",
    "firm_month_outcomes_extended_jan26.csv", low_memory=False
)
print("firm_master_ipcei:", firm_master_ipcei.shape)
print("firm_month_outcomes:", firm_month_outcomes.shape)

In [ ]:
# ============================================================
# Treatment prep + outcomes + consortium-size level outcome
# (mirrors 05a so every method shares one definition)
# ============================================================
fm = firm_master_ipcei.copy()
fm["first_ipcei_date"] = pd.to_datetime(fm["first_ipcei_date"], errors="coerce")
fm["first_ipcei_year"] = fm["first_ipcei_date"].dt.year

# Battery value chain 1 (pre-2020) excluded from treated AND controls
battery_round1 = (
    fm["ipcei_types"].astype(str).str.lower().str.contains("battery", na=False)
    & (fm["first_ipcei_year"] < 2020)
)
fm = fm[~battery_round1].copy()
fm["battery_ipcei"]  = fm["ipcei_types"].astype(str).str.lower().str.contains("battery", na=False).astype(int)
fm["hydrogen_ipcei"] = fm["ipcei_types"].astype(str).str.lower().str.contains("hydrogen", na=False).astype(int)
print(f"Battery IPCEI: {int(fm['battery_ipcei'].sum())}  Hydrogen IPCEI: {int(fm['hydrogen_ipcei'].sum())}")

# Outcomes: datetime month + integer counts
firm_month_outcomes["month"] = pd.to_datetime(firm_month_outcomes["month"], errors="coerce")
for col in [c for c in firm_month_outcomes.columns if c.startswith("n_collabs")]:
    firm_month_outcomes[col] = pd.to_numeric(firm_month_outcomes[col], errors="coerce").fillna(0).astype(int)

# Consortium-size level outcome (mirror 05a): mean firms/collaboration per firm-month
_pm = partnership_master_ipcei.copy()
_pm["post_date"] = pd.to_datetime(_pm["post_date"], errors="coerce")
_pm["month"] = _pm["post_date"].dt.to_period("M").dt.to_timestamp()
_endp = pd.concat([
    _pm[["collaboration_id", "month", "source"]].rename(columns={"source": "Linkedin_name"}),
    _pm[["collaboration_id", "month", "target"]].rename(columns={"target": "Linkedin_name"}),
], ignore_index=True).dropna(subset=["collaboration_id", "Linkedin_name"])
_nfirms = _endp.groupby("collaboration_id")["Linkedin_name"].nunique().rename("n_firms_in_collab")
_endp = (_endp.merge(_nfirms, on="collaboration_id", how="left")
              .dropna(subset=["month"])
              .drop_duplicates(subset=["Linkedin_name", "month", "collaboration_id"]))
mean_collab_size = (_endp.groupby(["Linkedin_name", "month"])["n_firms_in_collab"]
                         .mean().reset_index(name="mean_collab_size"))

# Covariates — identical to the PSM matching design
CONTINUOUS = ["log_staffCount", "age", "log_collab_count_2020"]
EXACT      = ["nace1_psm", "has_eu_operation"]
MAX_CONTROLS = 5000   # seeded random cap on control pool (computational; random != matching)

In [ ]:
# ============================================================
# Build unmatched eligible-pool panels (no matching)
# Eligibility uses only structural strata (EXACT) + not-beneficiary;
# baseline collaboration / centrality are NOT gated here (they enter
# the C&S estimator via xformla), so this is a genuine no-matching test.
# ============================================================
def build_unmatched(tech, treatment_var, date_col):
    pool = eligible_pool(
        fm, treatment_var, EXACT,
        treated_any_col="treated_any", exclude_other_treated=True,
        require_vars=CONTINUOUS, n_controls=MAX_CONTROLS,
    )
    glk = fm[["Linkedin_name", date_col]].rename(columns={date_col: "g"})
    cs = build_cs_panel(pool, firm_month_outcomes, glk, mean_collab_size=mean_collab_size)
    out = R_DATA_DIR / f"{tech}_unmatched_cs_ready.csv"
    cs.to_csv(out, index=False)
    print(f"{tech:8}: firms={pool['Linkedin_name'].nunique():>6} "
          f"(treated={int(pool.is_treated.sum())}, control={int(pool.is_control.sum())})  "
          f"-> {out.name}  [{len(cs):,} firm-months]")
    return cs

_ = build_unmatched("battery",  "battery_ipcei",  "first_battery_date")
_ = build_unmatched("hydrogen", "hydrogen_ipcei", "first_hydrogen_date")
print("\nDone. Consumed by 06/07_ck_didestimation_*_unmatched.Rmd (xformla adjustment).")